# Advanced Document OCR Pipeline

## Computer Vision: Enhance → Segment → Clean → Detect → Decide

This notebook demonstrates a complete document scanner pipeline with:
- Multiple enhancement methods (CLAHE, Gamma, HistEq, Denoising)
- Multi-strategy document detection (Contour, Hough, Edge Analysis)
- Morphological cleaning (closing, opening)
- Quality scoring and pass/fail decision logic
- Full pipeline visualization at every stage

In [ ]:
# Install dependencies
!pip install opencv-python pytesseract matplotlib numpy scikit-image

import sys
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from pipeline import run_pipeline, PipelineResult
from enhancement import apply_enhancement, ENHANCEMENT_METHODS
from detection import detect_edges, detect_document, apply_perspective_correction
from cleaning import binarize_document, apply_cleaning_pipeline
from decision import analyze_image_quality, compute_overall_quality, make_decision
from visualization import create_detailed_report, create_pipeline_summary

def show(img, title="", figsize=(12, 8)):
    plt.figure(figsize=figsize)
    if len(img.shape) == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title, fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def show2(img1, title1, img2, title2):
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    for ax, img, title in zip(axes, [img1, img2], [title1, title2]):
        if len(img.shape) == 2:
            ax.imshow(img, cmap='gray')
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=13)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

---
## Step 1: Load Input Image

In [ ]:
IMAGE_PATH = "input.jpg"

image = cv2.imread(IMAGE_PATH)
if image is None:
    raise FileNotFoundError(f"Could not load: {IMAGE_PATH}")

show(image, f"Original Image ({image.shape[1]}x{image.shape[0]})")

---
## Step 2: Enhancement (Choose method)

Available: `clahe`, `gamma`, `hist_eq`, `denoise`, `none`

In [ ]:
ENHANCEMENT_METHOD = "clahe"
UPSCALE_FACTOR = 2.0

enhanced = apply_enhancement(image, method=ENHANCEMENT_METHOD, upscale=UPSCALE_FACTOR)
show2(image, "Original", enhanced, f"Enhanced ({ENHANCEMENT_METHOD}, {UPSCALE_FACTOR}x upscale)")

---
## Step 3: Edge Detection (Segmentation)

In [ ]:
edges = detect_edges(enhanced)
show(edges, "Edge Detection (Canny with auto thresholds + dilation)")

---
## Step 4: Document Detection

Strategies: `contour`, `hough`, `edge`, or `auto` (tries all)

In [ ]:
DETECTION_STRATEGY = "auto"

corners, raw, strategy_used = detect_document(
    enhanced, edges, strategy=DETECTION_STRATEGY
)

if corners is not None:
    print(f"Document detected! Strategy: {strategy_used}")
    print(f"Corners:\n  TL: {corners[0]}\n  TR: {corners[1]}\n  BR: {corners[2]}\n  BL: {corners[3]}")
    
    # Visualize
    vis = enhanced.copy()
    labels = ["TL", "TR", "BR", "BL"]
    colors = [(0,220,0), (0,180,255), (0,0,255), (255,120,0)]
    for pt, name, c in zip(corners, labels, colors):
        pt_i = tuple(pt.astype(int))
        cv2.circle(vis, pt_i, 12, c, -1)
        cv2.putText(vis, name, (pt_i[0]+14, pt_i[1]-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, c, 2)
    cv2.polylines(vis, [corners.astype(np.int32)], True, (0, 255, 180), 2)
    show(vis, f"Detected Document Corners (strategy: {strategy_used})")
else:
    print("No document detected with any strategy.")

---
## Step 5: Perspective Correction (Cleaning)

In [ ]:
if corners is not None:
    corrected = apply_perspective_correction(enhanced, corners)
    show2(enhanced, "Before (distorted)", corrected, f"After (corrected {corrected.shape[1]}x{corrected.shape[0]})")
else:
    print("Skipping: no document detected")

---
## Step 6: Binarization (Thresholding)

In [ ]:
if corners is not None:
    binary = binarize_document(corrected, method="adaptive")
    show(binary, "Adaptive Thresholding Result")
else:
    print("Skipping: no document detected")

---
## Step 7: Morphological Cleaning

In [ ]:
if corners is not None:
    cleaned, clean_steps = apply_cleaning_pipeline(binary)
    
    show2(binary, "Before Cleaning", cleaned, "After Morphological Cleaning")
else:
    print("Skipping: no document detected")

---
## Step 8: Quality Analysis & Decision

In [ ]:
if corners is not None:
    metrics = analyze_image_quality(corrected, binary)
    quality_score = compute_overall_quality(metrics)
    grade, reason = make_decision(quality_score)
    
    print("=" * 50)
    print("QUALITY METRICS")
    print("=" * 50)
    for k, v in metrics.items():
        print(f"  {k.replace('_', ' ').title()}: {v:.2f}")
    
    print()
    print("=" * 50)
    print("FINAL DECISION")
    print("=" * 50)
    print(f"  Quality Score: {quality_score:.1f}%")
    print(f"  Grade: {grade}")
    print(f"  Reason: {reason}")
else:
    print("Skipping: no document detected")

---
## Full Pipeline (One-Click)

In [ ]:
result = run_pipeline(
    IMAGE_PATH,
    enhancement="clahe",
    detection_strategy="auto",
    upscale=2.0,
    output_dir="output",
    save_images=True,
)

print(f"Pipeline: {'SUCCESS' if result.success else 'FAILED'}")
print(f"Detection: {result.detection_strategy_used}")
print(f"Enhancement: {result.enhancement_method_used}")
print(f"Quality: {result.quality_score:.1f}%")
print(f"Grade: {result.decision_grade}")
print(f"Reason: {result.decision_reason}")

---
## Pipeline Summary Visualization

In [ ]:
if result.success:
    summary_path = create_pipeline_summary(result.stage_images, output_dir="output")
    from IPython.display import Image as IPImage
    display(IPImage(summary_path))

    print(f"All outputs saved to: output/visualization/")
    print(f"  - Each stage saved as PNG")
    print(f"  - Pipeline summary: pipeline_summary.png")
    print(f"  - Decision report: decision.txt")

---
## Bonus: Method Comparison

Runs all enhancement × detection combinations and reports best.

In [ ]:
from pipeline import run_comparison

print("Running full method comparison (this may take a moment)...")
all_results = run_comparison(IMAGE_PATH, output_dir="output/comparison")

print("\n" + "=" * 70)
print(f"{'Method':30s} {'Status':6s} {'Score':8s} {'Grade':25s}")
print("=" * 70)
best_combo, best_score = None, -1
for combo, res in sorted(all_results.items()):
    score = f"{res.quality_score:.1f}%" if res.success else "N/A"
    print(f"{combo:30s} {'OK' if res.success else 'FAIL':6s} {score:8s} {res.decision_grade:25s}")
    if res.success and res.quality_score > best_score:
        best_score = res.quality_score
        best_combo = combo

if best_combo:
    print(f"\nBest combination: {best_combo} (score: {best_score:.1f}%)")

---
## Summary

| Stage | Method | Status |
|-------|--------|--------|
| Enhancement | CLAHE / Gamma / HistEq / Denoise | ✓ |
| Segmentation | Canny Edge Detection | ✓ |
| Cleaning | Morphological (Close + Open) | ✓ |
| Detection | Contour / Hough / Edge Analysis | ✓ |
| Decision | Quality Score + Pass/Fail | ✓ |
| Bonus | Method Comparison / Gradio UI / Video | ✓ |